# Exploration

## Chargement et analyse rapide des données

Pour ce projet nous avons accès à un dossier contenant les csv "articles_metadata" et "clicks_sample" contenant respectivement des informations sur les articles et les clics (les clics fait pour accéder aux articles). Il contient aussi un dossier "click" contenant les csv des clics effectués chaque heures. Et enfin il contient un embedding déjà fait des articles au format pickle.

Commençons par importer et afficher les deux csv principaux.

In [1]:
import polars as pl
from pathlib import Path

DATA_PATH = Path("../data/news-portal-user-interactions-by-globocom/")

artcl_df = pl.read_csv(DATA_PATH / "articles_metadata.csv")
clk_df = pl.read_csv(DATA_PATH / "clicks_sample.csv")

print(artcl_df.shape)
artcl_df.head()

(364047, 5)


article_id,category_id,created_at_ts,publisher_id,words_count
i64,i64,i64,i64,i64
0,0,1513144419000,0,168
1,1,1405341936000,0,189
2,1,1408667706000,0,250
3,1,1408468313000,0,230
4,1,1407071171000,0,162


Le csv des articles contient des informations sur la catégorie dans laquelle est l'article, la date de sa création, l'identifiant du publieur et le nombre de mots qu'il contient.

In [2]:
print(clk_df.shape)
clk_df.head()

(1883, 12)


user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,1506825423271737,1506825423000,2,157541,1506826828020,4,3,20,1,20,2
0,1506825423271737,1506825423000,2,68866,1506826858020,4,3,20,1,20,2
1,1506825426267738,1506825426000,2,235840,1506827017951,4,1,17,1,16,2
1,1506825426267738,1506825426000,2,96663,1506827047951,4,1,17,1,16,2
2,1506825435299739,1506825435000,2,119592,1506827090575,4,1,17,1,24,2


Le csv des clics contient plusieurs colonnes que je ne comprend pas complétement encore et sur lesquels je vais devoir chercher plus de détails.

Regardons maintenant un exemple d'un csv contenu dans le dossier clicks.

In [3]:
clk_h0_df = pl.read_csv(DATA_PATH / "clicks/clicks_hour_000.csv")

print(clk_h0_df.shape)
clk_h0_df.head()

(1883, 12)


user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,1506825423271737,1506825423000,2,157541,1506826828020,4,3,20,1,20,2
0,1506825423271737,1506825423000,2,68866,1506826858020,4,3,20,1,20,2
1,1506825426267738,1506825426000,2,235840,1506827017951,4,1,17,1,16,2
1,1506825426267738,1506825426000,2,96663,1506827047951,4,1,17,1,16,2
2,1506825435299739,1506825435000,2,119592,1506827090575,4,1,17,1,24,2


En regardant le premier csv contenu dans clicks, je comprend qu'il est enfaite le même que clicks_sample et que ce dernier est juste un exemple du dossier. Affichons le deuxième pour comparer.

In [4]:
clk_h1_df = pl.read_csv(DATA_PATH / "clicks/clicks_hour_001.csv")

print(clk_h1_df.shape)
clk_h1_df.head()

(1415, 12)


user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
707,1506828988347444,1506828988000,3,119592,1506828988615,4,3,2,1,21,5
707,1506828988347444,1506828988000,3,202436,1506829330533,4,3,2,1,21,1
707,1506828988347444,1506828988000,3,237620,1506829360533,4,3,2,1,21,1
708,1506828991788445,1506828991000,2,68866,1506829050054,2,4,2,1,8,2
708,1506828991788445,1506828991000,2,96663,1506829080054,2,4,2,1,8,2


Il sera plus facile de manipuler un grand csv plutôt que plusieurs petit, nous allons donc fusionner tous les csv de clics.

In [5]:
clks_df = pl.scan_csv(DATA_PATH / "clicks").collect()

print(clks_df.shape)
clks_df.head()

(2988181, 12)


user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
str,str,str,str,str,str,str,str,str,str,str,str
"""0""","""1506825423271737""","""1506825423000""","""2""","""157541""","""1506826828020""","""4""","""3""","""20""","""1""","""20""","""2"""
"""0""","""1506825423271737""","""1506825423000""","""2""","""68866""","""1506826858020""","""4""","""3""","""20""","""1""","""20""","""2"""
"""1""","""1506825426267738""","""1506825426000""","""2""","""235840""","""1506827017951""","""4""","""1""","""17""","""1""","""16""","""2"""
"""1""","""1506825426267738""","""1506825426000""","""2""","""96663""","""1506827047951""","""4""","""1""","""17""","""1""","""16""","""2"""
"""2""","""1506825435299739""","""1506825435000""","""2""","""119592""","""1506827090575""","""4""","""1""","""17""","""1""","""24""","""2"""


On se retrouve donc avec un csv articles contenant 364 047 lignes et 5 colonnes et un csv clics contenant 2 988 181 lignes et 12 colonnes.

## Analyse poussée

On remarque que le tableau clics contient que des str, alors que les données sont en int normalement. Tentons de les convertirs, si des valeurs sont inadéquates à ce type, nous aurons une erreur.

In [6]:
clks_df = clks_df.cast(pl.Int64)

Parfait ça à fonctionner donc toutes nos valeurs sont bien compatibles int. Maintenant passons à l'analyse, regardons d'abord le nombre d'utilisateurs et d'articles dans chacun des tableaux.

In [7]:
print("Nombre d'utilisateur différent:", clks_df["user_id"].n_unique())
print("Nombre d'article différent dans le tableau clic:", clks_df["click_article_id"].n_unique())
print("Nombre d'article différent dans le tableau article:", artcl_df["article_id"].n_unique())

Nombre d'utilisateur différent: 322897
Nombre d'article différent dans le tableau clic: 46033
Nombre d'article différent dans le tableau article: 364047


En regardant ces données on se rend compte qu'il y a à peu près 8 fois plus d'articles que d'article ayant été consultés. Il faudra donc faire attention de ne pas rentrer dans le piège du biais de popularité. C'est à dire que si nous donnons trop d'importance aux nombre de consultations des articles dans notre algorithme de recommendation, les même articles seront consultés en boucle, ne laissant pas leur chance aux articles manquant de popularités.

Regardons maintenant la distribution d'autres données pertinentes.

In [ ]:
import plotly.express as xp
import plotly.io as pio
import plotly.graph_objects as go

pio.templates["center_title"] = go.layout.Template(
    layout=go.Layout(title_x=0.45, title_font={"size":28})
)

pio.templates.default += "+center_title" #type:ignore

xp.histogram(clks_df["session_size"], title="Distribution de la taille des sessions")


On remarque que la très grande majorité des utilisateurs ne lisent que quelques articles, et que de très rares utilisateurs en lisent beaucoup plus mais ça reste très rares. Ce comportement paraît humain, nous n'allons donc pas les supprimer.